# B2.5 · Feasibility filtering and reachability

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.4 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**.

| | |
|---|---|
| Tools used | CodeQL, tree-sitter, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Build a call graph from entry points and partition findings into reachable, unreachable and unknown.

**Why a security engineer needs it.** A finding in dead code costs the same to triage as one on the login path. The control it builds is: stage 10: decide whether an external caller can actually reach the sink before anyone is paged.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The finding is real. The code is dead. Reachability is the difference between a queue an engineer works and a queue an engineer learns to ignore, and it is the single largest false-positive killer in the pipeline.

> **At CyberTravels.** The finding is real and the code is unreachable from any traveller input. Reachability is what stops CyberTravels' queue becoming something engineers learn to ignore.

## 2 · The framework

```
   is there a path from untrusted input to this line?

   HTTP handler --> parse() --> validate() --> build_query() --> DB
                                                    ^
                                              the finding

   reachable   -> a finding
   unreachable -> a note

   the largest single false-positive killer in the pipeline
```

**Stage 10 — Feasibility filtering.** The last stage of Phase 3, and the one
that decides whether anyone gets paged.

A verified finding is a real bug in the code. It is not necessarily a real risk,
because the code may be unreachable: dead code, a test fixture, an internal
function no external caller can drive, a branch behind a feature flag that has
been off for two years.

Triaging an unreachable finding costs exactly as much as triaging one on the
login path, and there are usually far more of them. So this stage partitions
findings into three buckets — and the third bucket is the honest one:

- **reachable** — a path exists from an untrusted entry point to the sink,
- **unreachable** — no path exists,
- **unknown** — the analysis cannot decide, usually because of dynamic dispatch,
  reflection, or a framework that wires callers at runtime.

Reporting `unknown` as `unreachable` is how a pipeline quietly drops real bugs.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Where it breaks — collapsing `unknown` into `unreachable`

The tempting simplification. It makes the queue shorter and it is how real bugs get dropped, because dynamic dispatch is exactly where framework-wired handlers live.

## 4 · Phase 3 as a skill — and the counts that police it

Stages 7 to 10 only ever *shrink* the list. That is a property worth enforcing rather than trusting, so the skill's contract carries a `counts` object and the rule that it must never increase.

A pipeline whose `verified` count exceeds its `deduped` count has invented findings somewhere after the audit stage — and that is far easier to do by accident than it sounds, because a verification step that expands one finding per code path looks perfectly reasonable from the inside.

### The skill — [`skills/appsec/appsec-vuln-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-vuln-audit/SKILL.md)

```yaml
name: appsec-vuln-audit
description: >-
  Audit code for vulnerabilities against a threat model, then deduplicate,
  verify in context, and filter to what is actually reachable. Use when asked
  to review code for security bugs, run or interpret SAST, check whether a
  finding is a false positive, or reduce a noisy findings list to the ones
  worth a human's time.
allowed-tools: Read, Grep, Glob, Bash
```

# AppSec pipeline · Phase 3 — Analysis and filtering

Covers **stages 7–10**. This is where findings are produced — and, more
importantly, where most of them are thrown away.

The hard problem in application security is not finding candidate defects. It
is that a scanner emits hundreds and a human can act on ten. Every stage after
7 exists to shrink the list without losing the true positives.

## When to use this

When you have a threat model and a budget, or when handed a raw findings file
that nobody trusts. Stages 8–10 work on any findings list, including one from a
third-party scanner.

## Inputs

| Input | Required | Notes |
|---|---|---|
| `threat_model` + `plan.selected` | preferred | from appsec-threat-model |
| Source worktree | yes | verification needs the code, not just the finding |
| Existing findings | optional | run stages 8–10 alone to clean a noisy list |

## Procedure

**Stage 7 — Vulnerability auditing.** For each selected threat, examine the
path from entry to sink and decide whether the weakness is actually present.
Record for each finding: `cwe`, `file`, `line`, `unit`, the **evidence** (the
specific expression that is unsafe), and the **sanitiser** you looked for and
did not find. A finding that cannot name what was missing is a guess.

Three generations of analysis, and they are complementary, not competing:
grep-class pattern matching (fast, no dataflow), taint analysis (dataflow, no
semantics), and model-assisted review (semantics, no guarantees). Use the
cheapest one that can answer the question, and never let the third overrule the
second on a question of reachability — the model does not execute the program.

**Stage 8 — Deduplication.** The same defect appears many times: once per
scanner, once per path, once per call site. Collapse on the **defect identity**
— `(cwe, file, unit, sink_expression)` — not on the message text. Keep the
count: `occurrences` is signal about how exposed the defect is.

Match paths by parent directory plus filename tail. Deduplicating on a bare
basename silently merges two different files and loses a real finding.

**Stage 9 — Contextual verification.** For each surviving finding, look at the
surrounding code for the thing that makes it not-a-bug: a validator upstream, a
framework escaping the parameter, a type that cannot hold the payload, a caller
that only ever passes a constant. Record the verdict and the reason:
`confirmed`, `mitigated_by <what>`, or `needs_human`.

`needs_human` is a legitimate verdict and must stay available. A pipeline that
must decide will decide wrongly under uncertainty.

**Stage 10 — Feasibility filtering.** Drop what an attacker cannot actually
reach: code behind a feature flag that is off, an admin-only path in a service
with no admin, a sink whose input is fully constant. Record *why* each drop was
made, because the next scan will rediscover it and the reason is what stops
that work being repeated.

## Output contract

```json
{
  "findings": [
    {"id": "str", "cwe": "CWE-89", "file": "str", "line": 0, "unit": "str",
     "evidence": "str", "missing_control": "str",
     "occurrences": 1, "verdict": "confirmed|mitigated|needs_human",
     "verdict_reason": "str", "feasible": true, "confidence": 0.0}
  ],
  "dropped": [{"id": "str", "stage": 8, "why": "str"}],
  "counts": {"raw": 0, "deduped": 0, "verified": 0, "feasible": 0}
}
```

`counts` must be monotonically non-increasing across the four stages. If it is
not, the pipeline invented findings after the audit stage — stop and report.

## Failure modes

- **Confusing conformance with accuracy.** Output that matches this schema
  perfectly can still be entirely wrong. Schema validity is close to free;
  correctness is the expensive part. Never report conformance as a quality
  metric.
- **Dropping silently.** Every drop needs a stage and a reason.
- **Letting a model overrule dataflow on reachability.** It may propose a path;
  it may not confirm one.
- **Suppressing `needs_human` to look decisive.** Uncertainty that is hidden
  becomes someone's incident.

## Handoff

Feasible, confirmed findings go to **appsec-exploit-validate** for proof.
Everything else goes to **appsec-triage-report** with its verdict intact.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-vuln-audit/scripts/appsec_vuln_audit.py
SCRIPT = "skills/appsec/appsec-vuln-audit/scripts/appsec_vuln_audit.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 5 · Where it breaks — deduplicating on the wrong key

The skill says to collapse on the **defect identity**, `(cwe, file, unit, sink_expression)`, and never on the message text. Here is why that sentence is in the procedure.

## 6 · The same failure, from a real model

Everything above is constructed. Here is the identical failure produced by an actual open-weight model — **Moonlight-16B-A3B**, Moonshot AI's MoE from the Kimi team — run on a Kaggle CPU kernel against this skill's output contract.

It was given the contract and two vulnerable functions: an `open()` on a caller-supplied path, and an `os.system()` on a caller-supplied argument. Its answer is reproduced verbatim below ([full run](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/kimi/moonlight-16b-completion-prompt.txt)).

## 7 · Read that output again

It passes the contract with zero problems, and almost nothing in it is true.

## What you just proved

The call graph identifies three entry points, one of which uses dynamic dispatch. `load_report` is reachable, `debug_dump` and `legacy_export` are unknown rather than unreachable because runtime handler resolution cannot be ruled out. Two-bucket filtering silently drops both, and the three-bucket routing sends the unknowns to Phase 4 instead of paging or discarding them.

## Your turn

Count how many `unknown` cases your own reachability analysis produces, and find out what your tooling does with them. If it reports them as clean, the number of real bugs you are dropping is the size of that bucket.

---

**Next → [B2.6 · Sandbox replication](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*